In [ ]:
import QuantLib as ql
import torch as torch

from ir_vanilla_swap import IRVanillaSwap
from lgm_ir_monte_carlo import LGMIRMonteCarlo
from lgm_ir_utility import LGMIRUtility
from ir_bermudan_swaption import IRBermudanSwaption

## Set up underlying swap

In [ ]:
start_date = ql.Date(27, 12, 2018)
tenor = "10Y"
fixed_freq = "1Y"
float_freq = "6M"
calendar = ql.TARGET()
        
short_conv = ql.Following
long_conv = ql.ModifiedFollowing
date_gen_rule = ql.DateGeneration.Backward
end_of_month = False
fixed_dc = ql.Thirty360(ql.Thirty360.BondBasis)
float_dc = ql.Actual360()
notional = 1000000.0
fixed_rate = 0.03
pay_fixed = True

In [ ]:
ir_swap = IRVanillaSwap(start_date,
                                tenor,
                                fixed_freq,
                                float_freq,
                                calendar,
                                short_conv,
                                long_conv,
                                date_gen_rule,
                                end_of_month,
                                fixed_dc,
                                float_dc,
                                notional,
                                fixed_rate,
                                pay_fixed)

## Create Bermudan

In [ ]:
ex_schedule = [ql.Date(27, 12, 2019), ql.Date(27, 12, 2020), ql.Date(27, 12, 2021), ql.Date(27, 12, 2022),
                 ql.Date(27, 12, 2023), ql.Date(27, 12, 2024), ql.Date(27, 12, 2025), ql.Date(27, 12, 2026),
                 ql.Date(27, 12, 2027)]
longshort = True
ir_bermudan = IRBermudanSwaption(ir_swap, ex_schedule, longshort)

## Create an LGMIRUtility object

In [ ]:
dates = [ql.Date(27, 12, 2018), ql.Date(27, 12, 2020), ql.Date(27, 12, 2022), ql.Date(27, 12, 2024),
                 ql.Date(27, 12, 2026), ql.Date(27, 12, 2028)]
phi = (0.01, 0.02, 0.03, 0.04, 0.05)
psi = (1.0, 1.0, 1.0, 1.0, 1.0)
dtype = torch.float32

start_date = ql.Date(27, 12, 2018)
end_date = ql.Date(27, 12, 2028)

lgmirutility = LGMIRUtility(dates, phi, psi, start_date, end_date, dtype)

# Create the yield curves

In [ ]:
yield_curves = dict()
dc = ql.Actual365Fixed()
rate0d = ql.SimpleQuote(0.03)
rate_handle0d = ql.QuoteHandle(rate0d)
yts0d = ql.FlatForward(start_date, rate_handle0d, dc)
yts0d.enableExtrapolation()
hyts0d = ql.RelinkableYieldTermStructureHandle(yts0d)
yield_curves["0D"] = hyts0d

rate6M = ql.SimpleQuote(0.035)
rate_handle6M = ql.QuoteHandle(rate6M)
yts6M = ql.FlatForward(start_date, rate_handle6M, dc)
yts6M.enableExtrapolation()
hyts6M = ql.RelinkableYieldTermStructureHandle(yts6M)
yield_curves["6M"] = hyts6M

## Set up the device

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda', 0)
else:
    device = torch.device('cpu', 0)

In [ ]:
print(device)

 ## MC Params 1


In [ ]:
mc_params1 = dict()
mc_params1["no_paths"] = 2**14
mc_params1["seed"] = 98765
mc_params1["calendar"] = ql.UnitedStates(ql.UnitedStates.NYSE)

## LS Regression

In [ ]:
value, lsdata = ir_bermudan.value_LS(5, lgmirutility, yield_curves, mc_params1, device, dtype, start_date)

In [ ]:
value

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def plot_cont_infer_par(intrinsic_values, pred_values, par_rates, filename):
    plt.figure(figsize=(10,6)) 
    plt.scatter(par_rates, intrinsic_values, c='gray')
    plt.scatter(par_rates, pred_values, marker='x', c='k')
    plt.legend(["continuation values at t+1", "regression"])
    plt.xlabel("State at t")
    plt.ylabel("NPV")
    plt.savefig(filename, dpi=300, bbox_inches='tight')

In [ ]:
lsdict = lsdata[-1]
intrinsic_values = lsdict['cont_value']
pred_values = lsdict['inferred_value']
par_rates = lsdict['par_rates']

In [ ]:
filename_1 = "LSReg1.png"

In [ ]:
plot_cont_infer_par(intrinsic_values, pred_values, par_rates, filename_1)

In [ ]:
lsdict = lsdata[-2]
intrinsic_values = lsdict['cont_value']
pred_values = lsdict['inferred_value']
par_rates = lsdict['par_rates']

In [ ]:
filename_2 = "LSReg2.png"

In [ ]:
plot_cont_infer_par(intrinsic_values, pred_values, par_rates, filename_2)

In [ ]:
lsdict = lsdata[-3]
intrinsic_values = lsdict['cont_value']
pred_values = lsdict['inferred_value']
par_rates = lsdict['par_rates']

In [ ]:
filename_3 = "LSReg3.png"

In [ ]:
plot_cont_infer_par(intrinsic_values, pred_values, par_rates, filename_3)

In [ ]:
lsdict = lsdata[0]
intrinsic_values = lsdict['cont_value']
pred_values = lsdict['inferred_value']
par_rates = lsdict['par_rates']

In [ ]:
filename_4 = "LSReg4.png"

In [ ]:
plot_cont_infer_par(intrinsic_values, pred_values, par_rates, filename_4)

## DNN

In [ ]:
dnn_params = dict()
dnn_params["batch_size"] = 1024
dnn_params["test_split"] = 0.25
dnn_params["epochs"] = 20
dnn_params["learning_rate"] = 0.1
dnn_params["order"] = 5
dnn_params["early_stop_patience"] = 20

class NeuralNetDeep(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(NeuralNetDeep, self).__init__()
        self.fc1 = torch.nn.Linear(input_size, hidden_size)
        self.relu = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(hidden_size, hidden_size)
        self.relu2 = torch.nn.ReLU()
        self.fc3 = torch.nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu2(out)
        out = self.fc3(out)
        return out

In [ ]:
nn_model = NeuralNetDeep(dnn_params["order"], 32, 1)

value, dnn_data = ir_bermudan.value_DNN_multi_Reuse(lgmirutility, yield_curves, mc_params1, device,
                                              dtype, start_date, nn_model, dnn_params )

In [ ]:
dnndict = dnn_data[-1]
intrinsic_values = dnndict['cont_value']
pred_values = dnndict['inferred_value']
par_rates = dnndict['par_rates']
loss_data = dnndict['loss_data']

In [ ]:
filename_dnn_1 = "DNNReg1.png"

In [ ]:
plot_cont_infer_par(intrinsic_values, pred_values, par_rates, filename_dnn_1)

In [ ]:
def plot_loss(loss_data, filename):
    # Create count of the number of steps
    step_count = len(loss_data)
    loss_values = list()
    test_loss_values = list()
    step_data = [i for i in range(step_count)]
    
    # Get training and test loss histories
    for iloss in loss_data:
        loss_values.append(iloss['loss'])
        test_loss_values.append(iloss['test_loss'])

    # Visualize loss history
    plt.figure(figsize=(10,6)) 
    plt.plot(step_data, loss_values, 'k--')
    plt.plot(step_data, test_loss_values, 'k-')
    plt.legend(['Training Loss', 'Test Loss'])
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show();

In [ ]:
filename_dnnloss_1 = "DNNReg1_loss.png"

In [ ]:
plot_loss(loss_data, filename_dnnloss_1)

In [ ]:
dnndict = dnn_data[-2]
intrinsic_values = dnndict['cont_value']
pred_values = dnndict['inferred_value']
par_rates = dnndict['par_rates']
loss_data = dnndict['loss_data']

In [ ]:
filename_dnn_2 = "DNNReg2.png"

In [ ]:
plot_cont_infer_par(intrinsic_values, pred_values, par_rates, filename_dnn_2)

In [ ]:
filename_dnnloss_2 = "DNNReg2_loss.png"

In [ ]:
plot_loss(loss_data, filename_dnnloss_2)

In [ ]:
dnndict = dnn_data[-3]
intrinsic_values = dnndict['cont_value']
pred_values = dnndict['inferred_value']
par_rates = dnndict['par_rates']
loss_data = dnndict['loss_data']

In [ ]:
filename_dnn_3 = "DNNReg3.png"

In [ ]:
plot_cont_infer_par(intrinsic_values, pred_values, par_rates, filename_dnn_3)

In [ ]:
filename_dnnloss_3 = "DNNReg3_loss.png"

In [ ]:
plot_loss(loss_data, filename_dnnloss_3)

In [ ]:
dnndict = dnn_data[0]
intrinsic_values = dnndict['cont_value']
pred_values = dnndict['inferred_value']
par_rates = dnndict['par_rates']
loss_data = dnndict['loss_data']

In [ ]:
filename_dnn_4 = "DNNReg4.png"

In [ ]:
plot_cont_infer_par(intrinsic_values, pred_values, par_rates, filename_dnn_4)

In [ ]:
filename_dnnloss_4 = "DNNReg4_loss.png"

In [ ]:
plot_loss(loss_data, filename_dnnloss_4)

In [ ]:
value

### Neural Network using ZCBs

In [ ]:
hidden_layers = 2
layerwidth = 32
dnn_params = dict()
dnn_params["batch_size"] = 1024
dnn_params["test_split"] = 0.25
dnn_params["epochs"] = 20
dnn_params["learning_rate"] = 0.1

value, dnn_data = ir_bermudan.value_DNN_bondregress(lgmirutility, yield_curves, mc_params1, device,
                                                    dtype, start_date, hidden_layers, layerwidth, dnn_params)

In [ ]:
dnndict = dnn_data[-1]
intrinsic_values = dnndict['cont_value']
pred_values = dnndict['inferred_value']
par_rates = dnndict['par_rates']
loss_data = dnndict['loss_data']

filename_dnn_1 = "DNNZCBReg1.png"

plot_cont_infer_par(intrinsic_values, pred_values, par_rates, filename_dnn_1)

In [ ]:
filename_dnnloss_1 = "DNNZCBReg1_loss.png"

plot_loss(loss_data, filename_dnnloss_1)

In [ ]:
dnndict = dnn_data[-2]
intrinsic_values = dnndict['cont_value']
pred_values = dnndict['inferred_value']
par_rates = dnndict['par_rates']
loss_data = dnndict['loss_data']

filename_dnn_2 = "DNNZCBReg2.png"

plot_cont_infer_par(intrinsic_values, pred_values, par_rates, filename_dnn_2)

In [ ]:
filename_dnnloss_2 = "DNNZCBReg2_loss.png"

plot_loss(loss_data, filename_dnnloss_2)

In [ ]:
dnndict = dnn_data[-3]
intrinsic_values = dnndict['cont_value']
pred_values = dnndict['inferred_value']
par_rates = dnndict['par_rates']
loss_data = dnndict['loss_data']

filename_dnn_3 = "DNNZCBReg3.png"

plot_cont_infer_par(intrinsic_values, pred_values, par_rates, filename_dnn_3)

In [ ]:
filename_dnnloss_3 = "DNNZCBReg3_loss.png"

plot_loss(loss_data, filename_dnnloss_3)

In [ ]:
dnndict = dnn_data[0]
intrinsic_values = dnndict['cont_value']
pred_values = dnndict['inferred_value']
par_rates = dnndict['par_rates']
loss_data = dnndict['loss_data']

filename_dnn_4 = "DNNZCBReg4.png"

plot_cont_infer_par(intrinsic_values, pred_values, par_rates, filename_dnn_4)

In [ ]:
filename_dnnloss_4 = "DNNZCBReg4_loss.png"

plot_loss(loss_data, filename_dnnloss_4)

In [ ]:
value